## Data Analytics Module 2 - Apache Spark Manipulations

### Loading CSV files into the Unity Catalog Volume

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS spark_lab

In [0]:
import requests

# Define the current catalog
catalog_name = spark.sql("SELECT current_catalog()").collect()[0][0]

# Define the base path using the current catalog
volume_base = f"/Volumes/{catalog_name}/default/spark_lab"

# List of files to download
files = ["2019_edited.csv", "2020_edited.csv", "2021_edited.csv"]

# Download each file
for file in files:
    url = f"https://raw.githubusercontent.com/kuljotSB/DatabricksGenAIEngineer/refs/heads/main/Databricks_Fundamentals/{file}"
    response = requests.get(url)
    response.raise_for_status()

    # Write to Unity Catalog volume
    with open(f"{volume_base}/{file}", "wb") as f:
        f.write(response.content)

### Loading the CSV files into a dataframe

In [0]:
df = spark.read.load(f'/Volumes/{catalog_name}/default/spark_lab/*_edited.csv', format='csv')
display(df.limit(100))

### Defining Schema for the Dataframe

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
orderSchema = StructType([
    StructField("SalesOrderNumber", StringType()),
    StructField("SalesOrderLineNumber", IntegerType()),
    StructField("OrderDate", DateType()),
    StructField("CustomerName", StringType()),
    StructField("Email", StringType()),
    StructField("Item", StringType()),
    StructField("Quantity", IntegerType()),
    StructField("UnitPrice", FloatType()),
    StructField("Tax", FloatType())
])
df = spark.read.load(f'/Volumes/{catalog_name}/default/spark_lab/*_edited.csv', format='csv', schema=orderSchema)
display(df.limit(100))

### Cleaning the Data

In [0]:
from pyspark.sql.functions import col
df = df.dropDuplicates()
df = df.withColumn('Tax', col('UnitPrice') * 0.08)
df = df.withColumn('Tax', col('Tax').cast("float"))

### Creating a new Dataframe

In [0]:
customers_df = df.select("CustomerName", "Email", "Item", "Quantity")
customers_df = customers_df.withColumn("FirstName", split(customers_df["CustomerName"], " ").getItem(0))
customers_df = customers_df.withColumn("LastName", split(customers_df["CustomerName"], " ").getItem(1))

display(customers_df)

### Creating distinct customer entries

In [0]:
print(customers_df.count())
print(customers_df.distinct().count())

### Creating a Product Sales Dataframe

In [0]:
productSales = df.select("Item", "Quantity").groupBy("Item").sum()
display(productSales)

### Aggregating Yearly Sales

In [0]:
yearlySales = df.select(year("OrderDate").alias("Year")).groupBy("Year").count().orderBy("Year")
display(yearlySales)